In [ ]:
"""
Figure: Heatwave Days (Historical vs Hist-nat)

**Authors:** Emily Vosper + Ruby Leiber  
**Affiliation:** University of Bristol  
**Date:** 26/06/2025  

**Description:** Plots the difference in heatwave days and extreme heat days between historical and hist-nat simulations over Brazil, 2000–2018.
"""

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import PowerNorm, LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable, get_cmap
from scipy.stats import linregress, pearsonr
import pyreadr

In [ ]:
base_path = '/bp1/geog-tropical/data/BREATHE/timeseries_igr_new'

models = {
    'ACCESS-CM2':      [f'r{r}i1p1f1' for r in [1]],
    'ACCESS-ESM1-5':   [f'r{r}i1p1f1' for r in range(1, 4)],
    'BCC-CSM2-MR':     [f'r{r}i1p1f1' for r in [1]],
    'CNRM-CM6-1':      [f'r{r}i1p1f2' for r in range(1, 4)],
    'CanESM5':         [f'r{r}i1p1f1' for r in range(1, 26) if r not in [12, 16, 17, 18, 19, 20, 21, 24]],
    'CanESM5-p2':      [f'r{r}i1p2f1' for r in range(1, 26) if r not in [3, 7, 12, 13, 14, 15, 17, 19, 23]],
    'FGOALS-g3':       [f'r{r}i1p1f1' for r in [1]],
    'GFDL-CM4':        [f'r{r}i1p1f1' for r in [1]],
    'GFDL-ESM4':       [f'r{r}i1p1f1' for r in [1]],
    'HadGEM3-GC31-LL': [f'r{r}i1p1f3' for r in range(1, 61) if r not in [6, 7, 8, 9, 10]],
    'IPSL-CM6A-LR':    [f'r{r}i1p1f1' for r in range(1, 7) if r != 3],
    'MIROC6':          [f'r{r}i1p1f1' for r in range(1, 3)],
    'MRI-ESM2-0':      [f'r{r}i1p1f1' for r in range(1, 6) if r != 4],
    'NorESM2-LM':      [f'r{r}i1p1f1' for r in range(1, 4)],
}

In [ ]:
polys = '/bp1/geog-tropical/data/BREATHE/RG2017_rgi_20180911/RG2017_rgi.shp'
gdf = gpd.read_file(polys)
full_rgis = sorted(gdf['rgi'].astype(str).unique())

In [ ]:
def load_ts(path):
    df = pd.read_parquet(path)
    df = df.set_index(['date', 'rgi']).sort_index()
    return df.loc['2000':'2018']

parquets = {}

for model, runs in models.items():
    for run in runs:
        if model == 'CanESM5-p2':
            hist_file = f"{base_path}/tas_CanESM5_historical_{run}_timeseries_2000_2020.parquet"
            nat_file  = f"{base_path}/tas_CanESM5_hist-nat_{run}_timeseries_2000_2020.parquet"
        else:
            hist_file = f"{base_path}/tas_{model}_historical_{run}_timeseries_2000_2020.parquet"
            nat_file  = f"{base_path}/tas_{model}_hist-nat_{run}_timeseries_2000_2020.parquet"

        parquets[f"{model}_hist_{run}"]     = load_ts(hist_file)
        parquets[f"{model}_hist-nat_{run}"] = load_ts(nat_file)

In [ ]:
def calc_gdf_mean(gdfs):
    """Average daily_mean_temperature across a list of GeoDataFrames."""
    temps = np.stack([gdf['daily_mean_temperature'].values for gdf in gdfs], axis=0)
    gdf_mean = gdfs[0].copy()
    gdf_mean['daily_mean_temperature'] = temps.mean(axis=0)
    return gdf_mean


def process_pq(df):
    """Group by rgi, compute mean temperature, and merge with shapefile."""
    df = df.reset_index()
    df_grouped = df.groupby('rgi', as_index=False)['daily_mean_temperature'].mean()
    df_merged = pd.merge(df_grouped, gdf, on='rgi', how='left')
    return gpd.GeoDataFrame(df_merged, geometry='geometry', crs=gdf.crs)


def count_heatwaves(hot_days_bool):
    """Count heatwave events (runs of >= 3 consecutive hot days)."""
    count, run_length = 0, 0
    for val in hot_days_bool:
        if val:
            run_length += 1
        else:
            if run_length >= 3:
                count += 1
            run_length = 0
    if run_length >= 3:
        count += 1
    return count


def calc_heatwave_days(hist_pqs, hist_nat_pqs, full_rgis):
    """Compute heatwave counts per IGR for hist and hist-nat ensembles."""
    hist_hw_counts, hist_nat_hw_counts = [], []

    for pq in hist_pqs:
        pq = pq.reset_index()
        percentiles = pq.groupby('rgi')['daily_mean_temperature'].apply(
            lambda x: np.percentile(x, 90)
        )
        def count_hw(state):
            threshold = percentiles[state.name]
            return count_heatwaves(state['daily_mean_temperature'].values > threshold)
        hist_hw_counts.append(pq.groupby('rgi').apply(count_hw).reindex(full_rgis))

    for pq in hist_nat_pqs:
        pq = pq.reset_index()
        hist_nat_hw_counts.append(pq.groupby('rgi').apply(count_hw).reindex(full_rgis))

    return np.array(hist_hw_counts), np.array(hist_nat_hw_counts)


def calc_extreme_heat_days(hist_pqs, hist_nat_pqs, mmt):
    """Count days exceeding MMT threshold per IGR for hist and hist-nat."""
    hist_eh_days, hist_nat_eh_days = [], []

    def count_eh(state):
        return np.sum(state['daily_mean_temperature'].values > state['MMT'].iloc[0])

    for pq in hist_pqs:
        pq = pq.reset_index().merge(mmt[['rgi', 'MMT']], on='rgi', how='left')
        hist_eh_days.append(pq.groupby('rgi').apply(count_eh).reindex(full_rgis))

    for pq in hist_nat_pqs:
        pq = pq.reset_index().merge(mmt[['rgi', 'MMT']], on='rgi', how='left')
        hist_nat_eh_days.append(pq.groupby('rgi').apply(count_eh).reindex(full_rgis))

    return np.array(hist_eh_days), np.array(hist_nat_eh_days)

In [ ]:
mmt = pyreadr.read_r('two_stage_TS_RGI_cohort.RData')['mmt_blup']
mmt = mmt.rename(columns={'COD_RGI': 'rgi'})
mmt['rgi'] = mmt['rgi'].astype(str)

In [ ]:
model_hist_hws, model_hist_nat_hws = [], []
model_hist_ehs, model_hist_nat_ehs = [], []

for model, runs in models.items():
    hist_pqs     = [parquets[f"{model}_hist_{run}"]     for run in runs]
    hist_nat_pqs = [parquets[f"{model}_hist-nat_{run}"] for run in runs]

    hist_hw, hist_nat_hw = calc_heatwave_days(hist_pqs, hist_nat_pqs, full_rgis)
    hist_eh, hist_nat_eh = calc_extreme_heat_days(hist_pqs, hist_nat_pqs, mmt)

    model_hist_hws.append(hist_hw.mean(axis=0))
    model_hist_nat_hws.append(hist_nat_hw.mean(axis=0))
    model_hist_ehs.append(hist_eh.mean(axis=0))
    model_hist_nat_ehs.append(hist_nat_eh.mean(axis=0))

# Multi-model means
tas_hist_hws        = np.array(model_hist_hws).mean(axis=0)
tas_hist_nat_hws    = np.array(model_hist_nat_hws).mean(axis=0)
tas_hist_ehdays     = np.array(model_hist_ehs).mean(axis=0)
tas_hist_nat_ehdays = np.array(model_hist_nat_ehs).mean(axis=0)

In [ ]:
heatwaves = pd.DataFrame({
    'rgi':                    full_rgis,
    'daily_mean_temperature': (tas_hist_hws - tas_hist_nat_hws) / 19,
    'heatwave_hist':          tas_hist_hws / 19,
    'heatwave_hist_nat':      tas_hist_nat_hws / 19,
})

extreme_heat_days = pd.DataFrame({
    'rgi':               full_rgis,
    'additional_eh_days': (tas_hist_ehdays - tas_hist_nat_ehdays) / 19,
    'heatwave_hist':     tas_hist_ehdays / 19,
    'heatwave_hist_nat': tas_hist_nat_ehdays / 19,
})

extreme_heat_days.to_csv('extreme_heat_days.csv', index=False)

hw     = process_pq(heatwaves)
hw_mun = hw  # alias used in plotting

In [ ]:
elev_df = pd.read_parquet('/bp1/geog-tropical/data/BREATHE/igr_elevation.parquet')
elev_grouped = elev_df.groupby('rgi', as_index=False)['mean_elevation'].mean()
df_gdf_state = gpd.GeoDataFrame(
    pd.merge(elev_grouped, gdf, on='rgi', how='left'),
    geometry='geometry', crs=gdf.crs
)

In [ ]:
# --- Colormaps ---
elev_cmap = LinearSegmentedColormap.from_list('elev_cmap', [
    '#88A280', '#E5B582', '#A07F61', '#786552'
])

vmin, vmax = 0, 4.5
power_norm = PowerNorm(gamma=3, vmin=vmin, vmax=vmax)

def warped_colormap(base_cmap_name, norm, N=256):
    base_cmap = get_cmap(base_cmap_name, N)
    linear_vals = np.linspace(norm.vmin, norm.vmax, N)
    return LinearSegmentedColormap.from_list('warped_cmap', base_cmap(norm(linear_vals)))

warped_cmap = warped_colormap('magma_r', power_norm)

# --- Figure layout ---
fig = plt.figure(figsize=(20, 16))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 2], height_ratios=[1.5, 1], wspace=0.2, hspace=0)

# === a. Elevation map ===
ax2 = fig.add_subplot(gs[0, 0])
df_gdf_state.plot(
    column='mean_elevation', ax=ax2, legend=True, cmap=elev_cmap,
    vmin=0, vmax=1000,
    legend_kwds={'shrink': 0.6, 'pad': -0.06, 'label': 'Mean Elevation (m)'}
)
ax2.axis('off')
ax2.text(0.01, 0.99, 'a.', transform=ax2.transAxes, fontsize=18, fontweight='bold', va='top')

# === b. Scatter: heatwaves vs elevation ===
ax3 = fig.add_subplot(gs[1, 0])
combined = hw_mun[['rgi', 'daily_mean_temperature']].merge(df_gdf_state[['rgi', 'mean_elevation']], on='rgi')

ax3.scatter(
    combined['daily_mean_temperature'], combined['mean_elevation'],
    s=0.5,
    c=combined['daily_mean_temperature'],
    cmap=warped_cmap, norm=Normalize(vmin=vmin, vmax=vmax),
    edgecolor=None
)



# Fit
coeffs = np.polyfit(combined['daily_mean_temperature'], combined['mean_elevation'], 1)
poly = np.poly1d(coeffs)

# Plot
x_range = np.linspace(combined['daily_mean_temperature'].min(),
                      combined['daily_mean_temperature'].max(), 200)
# ax3.plot(x_range, poly(x_range), color='#aaaaaa', linewidth=1.5)

# Use the same aligned data for Pearson
# r, p = pearsonr(combined['daily_mean_temperature'], combined['mean_elevation'])
# ax3.text(0.1, 0.7, f'Pearson r = {r:.2f}\np-value = {p:.3f}',
#          transform=ax3.transAxes, fontsize=12, va='top')

ax3.set_xlabel('Number of additional heatwaves per year', fontsize=14)
ax3.set_ylabel('Mean elevation (m)', fontsize=14)
ax3.set_xticks([ 1, 2, 3, 4])
ax3.set_yticks([0, 200, 400, 600, 800, 1000, 1200, 1400])
for spine in ['top', 'right']:
    ax3.spines[spine].set_visible(False)
ax3.spines['left'].set_bounds(0, 1400)
ax3.spines['bottom'].set_bounds(1, 4)
for spine in ['left', 'bottom']:
    ax3.spines[spine].set_linewidth(2)
ax3.tick_params(labelsize=12, length=8, width=2)
ax3.text(0.01, 1.06, 'b.', transform=ax3.transAxes, fontsize=18, fontweight='bold', va='top')

# === c. Heatwave map ===
ax1 = fig.add_subplot(gs[:, 1])
hw.plot(
    column='daily_mean_temperature',
    cmap=warped_cmap, norm=Normalize(vmin=vmin, vmax=vmax),
    ax=ax1, legend=False, alpha=0.9,
    edgecolor='white', linewidth=0.2
)

cities = [
    {'name': 'Rio de Janeiro', 'x': -43.189, 'y': -22.917, 'dx':  0.4, 'dy': -0.75, 'marker': 'o'},
    {'name': 'Brasília',       'x': -47.895, 'y': -15.807, 'dx':  0.4, 'dy': -0.75, 'marker': 's'},
    {'name': 'Salvador',       'x': -38.505, 'y': -12.986, 'dx':  0.4, 'dy': -0.75, 'marker': 'o'},
    {'name': 'São Paulo',      'x': -46.638, 'y': -23.568, 'dx':  0.3, 'dy': -1.0,  'marker': 'o'},
    {'name': 'Recife',         'x': -35.383, 'y':  -8.059, 'dx':  0.1, 'dy': -0.1,  'marker': 'o'},
    {'name': 'Fortaleza',      'x': -38.529, 'y':  -3.724, 'dx':  0.4, 'dy':  0.3,  'marker': 'o'},
    {'name': 'Manaus',         'x': -60.021, 'y':  -3.122, 'dx':  0.4, 'dy': -0.75, 'marker': 'o'},
    {'name': 'Curitiba',       'x': -49.265, 'y': -25.428, 'dx':  0.4, 'dy': -0.75, 'marker': 'o'},
    {'name': 'Porto Alegre',   'x': -51.211, 'y': -30.038, 'dx':  0.4, 'dy': -0.75, 'marker': 'o'},
]
for city in cities:
    ax1.scatter(city['x'], city['y'], color='black', s=10, marker=city['marker'])
    ax1.annotate(city['name'], xy=(city['x'] + city['dx'], city['y'] + city['dy']),
                 xycoords='data', fontsize=12)

ax1.axis('off')
ax1.text(0.05, 0.98, 'c.', transform=ax1.transAxes, fontsize=18, fontweight='bold', va='top')

sm = ScalarMappable(norm=Normalize(vmin=vmin, vmax=vmax), cmap=warped_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax1, ticks=np.arange(vmin, vmax + 1), shrink=0.6, pad=-0.02)
cbar.set_label('Change in yearly heatwaves')
cbar.ax.tick_params(color='#342020', labelcolor='#342020')

fig.subplots_adjust(top=0.7, bottom=0.2)
plt.savefig('heatwaves_hist_vs_nat.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
hw.to_csv('hw.csv', index=False)